# ViT-S/16: похожие классы в батчах

Новый старт с внешних pretrained-весов. Train 655500 / val 34500, прежняя фиксированная разбивка.

Батч 64: 48 обычных + 8/8 из похожей пары. 149 пар выбраны только на перенесённых train-примерах. Cross-entropy с label smoothing 0.05, AdamW, EMA 0.99, BF16.

Warmup половина эпохи, затем ReduceLROnPlateau по EMA accuracy: factor=0.5, patience=1, threshold=0.0005. Backbone LR 0.0001, head 0.001.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

output = Path('output/pretrained-vit-similar-95-20261015')
config = json.loads((output / 'config.json').read_text())
config

In [ ]:
scores = pd.read_csv(output / 'scores.csv')
display(scores)

In [ ]:
reference = json.loads((output / "reference.json").read_text())
display(pd.DataFrame(reference["references"]).T)


In [ ]:
history = pd.read_csv(output / 'history.csv')
display(history)

In [ ]:
summary = json.loads((output / 'summary.json').read_text())
summary

In [ ]:
verification = json.loads((output / 'verification.json').read_text())
assert verification['val_examples'] == 34500
assert verification['submission_rows'] == 172500
assert verification['submission_differences'] == 0
assert verification['accuracy'] == summary['selected']['accuracy']
verification

In [ ]:
submission = pd.read_csv(output / 'submission.csv')
assert len(submission) == 172500
assert submission['id'].tolist() == list(range(172500))
assert submission['label'].between(0, 344).all()
display(submission.head())

In [ ]:
baseline = pd.read_csv("output/pretrained-vit-95-20261015/history.csv")
comparison = history[["epoch", "accuracy", "ema_accuracy"]].merge(baseline[["epoch", "accuracy", "ema_accuracy"]], on="epoch", suffixes=("_similar", "_baseline"))
display(comparison)
